In [33]:
# importar bibliotecas
import pandas as pd
from sqlalchemy import create_engine

db_config = {'user': 'practicum_student',         # nome de usuário
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs',  # senha
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,                       # porta de conexão
             'db': 'data-analyst-final-project-db'}  # nome do banco de dados

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                         db_config['pwd'],
                                                         db_config['host'],
                                                         db_config['port'],
                                                         db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})


In [49]:
# Consulta SQL
sample_query = """SELECT * FROM books
LIMIT 5"""

# Lendo os dados da consulta e armazenando em um DataFrame
df = pd.io.sql.read_sql(sample_query, con=engine)

# Exibindo o DataFrame
print(df)


   book_id  author_id                                              title  \
0        1        546                                       'Salem's Lot   
1        2        465                 1 000 Places to See Before You Die   
2        3        407  13 Little Blue Envelopes (Little Blue Envelope...   
3        4         82  1491: New Revelations of the Americas Before C...   
4        5        125                                               1776   

   num_pages publication_date  publisher_id  
0        594       2005-11-01            93  
1        992       2003-05-22           336  
2        322       2010-12-21           135  
3        541       2006-10-10           309  
4        386       2006-07-04           268  


In [50]:
# Consulta SQL
sample_query = """SELECT * FROM authors
LIMIT 5"""

# Lendo os dados da consulta e armazenando em um DataFrame
df = pd.io.sql.read_sql(sample_query, con=engine)

# Exibindo o DataFrame
print(df)


   author_id                          author
0          1                      A.S. Byatt
1          2  Aesop/Laura Harris/Laura Gibbs
2          3                 Agatha Christie
3          4                   Alan Brennert
4          5        Alan Moore/David   Lloyd


In [51]:
# Consulta SQL
sample_query = """SELECT * FROM ratings
LIMIT 5"""

# Lendo os dados da consulta e armazenando em um DataFrame
df = pd.io.sql.read_sql(sample_query, con=engine)

# Exibindo o DataFrame
print(df)


   rating_id  book_id       username  rating
0          1        1     ryanfranco       4
1          2        1  grantpatricia       2
2          3        1   brandtandrea       5
3          4        2       lorichen       3
4          5        2    mariokeller       2


In [52]:
# Consulta SQL
sample_query = """SELECT * FROM reviews
LIMIT 5"""

# Lendo os dados da consulta e armazenando em um DataFrame
df = pd.io.sql.read_sql(sample_query, con=engine)

# Exibindo o DataFrame
print(df)


   review_id  book_id       username  \
0          1        1   brandtandrea   
1          2        1     ryanfranco   
2          3        2       lorichen   
3          4        3  johnsonamanda   
4          5        3    scotttamara   

                                                text  
0  Mention society tell send professor analysis. ...  
1  Foot glass pretty audience hit themselves. Amo...  
2  Listen treat keep worry. Miss husband tax but ...  
3  Finally month interesting blue could nature cu...  
4  Nation purpose heavy give wait song will. List...  


In [53]:
# Consulta SQL
sample_query = """SELECT * FROM publishers
LIMIT 5"""

# Lendo os dados da consulta e armazenando em um DataFrame
df = pd.io.sql.read_sql(sample_query, con=engine)

# Exibindo o DataFrame
print(df)


   publisher_id                          publisher
0             1                                Ace
1             2                           Ace Book
2             3                          Ace Books
3             4                      Ace Hardcover
4             5  Addison Wesley Publishing Company


### Relações entre os banco de dados

BOOKS se relaciona com AUTHORS através do author_id
BOOKS se relaciona com publishers através do publisher_id
BOOKS se relaciona com RATINGS e REVIEWS pelo book_id

Todos banco de dados estão relacionados, porém somente e unicamente com o BOOKS, todas consultas entre tabelas diferentes precisam passar pelo books

Encontre o número de livros publicados após 1º de janeiro de 2000.

In [39]:
# Consulta SQL
sample_query = """
SELECT COUNT(book_id) AS BOOKS_TOTAL 
FROM books  
WHERE publication_date::date > '2000-01-01'
"""

# Lendo os dados da consulta e armazenando em um DataFrame
df = pd.io.sql.read_sql(sample_query, con=engine)

# Exibindo o DataFrame
print(df)


   books_total
0          819


O catálogo é majoritariamente atual, pós anos 2000, tendo sido lançado 819 livros, ~82% do catálogo.

Encontre o número de resenhas de usuários e a nota média para cada livro.

In [40]:
# Consulta SQL
sample_query = """
SELECT 
    books.title AS BOOK_NAME,
    AVG(ratings.rating) AS AVG_RATE,
    COUNT(reviews.review_id) AS TOTAL_REVIEW
FROM ratings 
JOIN reviews ON ratings.book_id = reviews.book_id
JOIN books ON ratings.book_id = books.book_id
GROUP BY books.book_id
"""

# Lendo os dados da consulta e armazenando em um DataFrame
df = pd.io.sql.read_sql(sample_query, con=engine)

# Exibindo o DataFrame
print(df.sample(30))


                                             book_name  avg_rate  total_review
879  Fantastic Beasts and Where to Find Them (Hogwa...  3.888889            36
764                             The Brothers Karamazov  3.428571            28
210               His Majesty's Dragon (Temeraire  #1)  3.666667             9
339  Julie and Julia: 365 Days  524 Recipes  1 Tiny...  3.600000            20
435                                 A Room with a View  3.800000            15
664                             The Undomestic Goddess  4.222222            36
866                              The Mill on the Floss  3.000000             4
57                                          Baby Proof  3.600000            15
231                                 Sputnik Sweetheart  3.666667             9
256                                   The Pilot's Wife  3.333333             9
120                                     Norwegian Wood  3.888889            45
708                                 The Secret Histo

Há livros com muitas reviews e outros com poucas, o que pode implicar em uma enviesação na nota média, já que com poucas avaliações qualquer avaliaçao que se destaque das outras pode puxar a nota para cima ou para baixo

Identifique a editora que publicou o maior número de livros com mais de 50 páginas (excluindo folhetos).

In [41]:
# Consulta SQL
sample_query = """
SELECT 
    publishers.publisher AS PUBLISHER_NAME,
    COUNT(books.book_id) AS BOOKS_COUNT
FROM books 
JOIN publishers ON books.publisher_id = publishers.publisher_id
WHERE books.num_pages > 50
GROUP BY publishers.publisher_id
ORDER BY BOOKS_COUNT DESC
LIMIT 1
"""

# Lendo os dados da consulta e armazenando em um DataFrame
df = pd.io.sql.read_sql(sample_query, con=engine)

# Exibindo o DataFrame
print(df)


  publisher_name  books_count
0  Penguin Books           42


A editora que mais publicou livros foi a Penguin Books, que ao todo teve 42 publicações acima do limiar de 50 páginas

Identifique o autor com a nota média mais alta, considerando apenas livros com pelo menos 50 avaliações.

In [45]:
# Consulta SQL
sample_query = """
SELECT 
    authors.author AS AUTHOR,
    AVG(ratings.rating) AS AVG_RATING
FROM books 

JOIN authors ON books.author_id = authors.author_id
JOIN ratings ON books.book_id = ratings.book_id

GROUP BY authors.author_id

HAVING COUNT(ratings.rating_id) > 50

ORDER BY AVG_RATING DESC
LIMIT 1
"""

# Lendo os dados da consulta e armazenando em um DataFrame
df = pd.io.sql.read_sql(sample_query, con=engine)

# Exibindo o DataFrame
print(df)


                       author  avg_rating
0  J.K. Rowling/Mary GrandPré    4.288462


Considerando apenas os autores com livros que tenham um mínimo de 50 avaliações J.K. Rowling está em primeiro lugar de nota média, com 4.28 de um nota máxima de 5

Encontre o número médio de resenhas com texto entre usuários que avaliaram mais de 50 livros.

In [54]:
# Consulta SQL
sample_query = """
SELECT AVG(total_reviews) 
FROM (
    SELECT username, 
    COUNT(review_id) AS total_reviews
    FROM reviews
    WHERE username IN (SELECT username
                        FROM ratings
                        GROUP BY username
                        HAVING COUNT(book_id) > 50)
                        GROUP BY username
)
"""

# Lendo os dados da consulta e armazenando em um DataFrame
df = pd.io.sql.read_sql(sample_query, con=engine)

# Exibindo o DataFrame
print(df)


         avg
0  24.333333


A média de resenhas com texto para um usuário que deu avaliação por nota para pelo menos 50 livros é de ~24,3 resenhas por usuário, o que pode significar que um usuário mais ativo e engajado que está ativamente avaliando livros com nota está mais propenso a escrever resenhas de texto, o que pode implicar em esclarecer melhor para outros usuários a trama de um livro ou se vale ou não a pena comprar